In [ ]:
# Figure 4

In [8]:
from pathlib import Path
import re
import math
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

try:
    import geopandas as gpd
except ImportError as e:
    raise ImportError("This notebook needs geopandas. Install it with: pip install geopandas") from e

# =============================================================================
# GLOBAL FONT ENFORCEMENT
# =============================================================================
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix" 
plt.rcParams["axes.unicode_minus"] = False

# =============================================================================
# USER SETTINGS
# =============================================================================

# --- 1. Input & Output Directories ---
UHI_INPUT_DIR = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_4\Input_CSV\UHI")
HI_INPUT_DIR  = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_4\Input_CSV\HI")
INDIA_BOUNDARY_FILE = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_4\India_boundary\India_boundary.shp")
CITY_COORDS_CSV = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_4\city_coordinates\city_lat_lon.csv")

OUTPUT_DIR = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_4\output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. What to Plot & Layout Strategy ---
PLOT_OPTION = 1
FIGURE_LAYOUT = "all_sources_grid"
GRID_ORIENTATION = "seasons_rows_files_cols"
GLOBAL_SCALE_MODE = "all_metrics"
VALUE_COLUMN = "Mean"
SEASONS_TO_PLOT = ["MAM", "JJAS", "ON"]
METRICS_TO_PLOT = ["P90"]

# --- 3. Source Ordering & Labels ---
SOURCE_ORDER = ["historical_imdaa_1985-2014", "ssp245_near_2026-2055", "ssp585_near_2026-2055", "ssp245_far_2070-2099", "ssp585_far_2070-2099"]
FILE_LABELS = {
    "historical_imdaa_1985-2014": "Historical", 
    "ssp245_near_2026-2055": "SSP245 nf", 
    "ssp585_near_2026-2055": "SSP585 nf", 
    "ssp245_far_2070-2099": "SSP245 ff", 
    "ssp585_far_2070-2099": "SSP585 ff"
}
SOURCE_KEY_FILTER = None
FALLBACK_TO_ALL_FILES = True

# --- 4. Main Title & Outer Subplot Text Labels ---
MAIN_TITLE = "HI and UHI city seasonal summaries"
MAIN_TITLE_FONTSIZE = 18
MAIN_TITLE_BOLD = True

SHOW_PANEL_LABELS = True
PANEL_LABEL_POSITION = "upper left"
PANEL_LABEL_FONTSIZE = 22
PANEL_LABEL_BOLD = True

SHOW_SEASON_LABEL = True
SEASON_LABEL_POSITION = "upper right"
SEASON_LABEL_FONTSIZE = 20
SEASON_LABEL_BOLD = True

SHOW_FILE_LABELS = True
FILE_LABEL_POSITION = "top"
FILE_LABEL_FONTSIZE = 22
FILE_LABEL_BOLD = True

# --- 5. Aesthetics & HI (Heat Index) Color mapping ---
PRINT_GLOBAL_EXTREMES = True  
HI_FIXED_POINT_SIZE = 160     
HI_CMAP = "YlOrRd"            
UHI_REPRESENTATION_MODE = "symbols" 

UHI_RING_MIN_SIZE = 20
UHI_RING_MAX_SIZE = 700
UHI_RING_LEGEND_LEVELS = 11
UHI_RING_LEGEND_MARKERSIZE_SCALE = 0.42
UHI_RING_LEGEND_LINEWIDTH = 0.8

# --- 6. UHI Symbol Mapping Logic ---
UHI_SYMBOL_BINS = [-2, -1, 0, 1, 2, 3, 4, 5]
UHI_SYMBOL_MARKERS = ["s", "^", "*", "*", "^", "s", "o"]
UHI_SYMBOL_FACE_COLORS = ["white", "white", "white", "black", "black", "black", "black"]
UHI_SYMBOL_POINT_SIZE = 50
UHI_SYMBOL_BOLD = False
UHI_SYMBOL_EDGEWIDTH_BOLD = 0.8            
UHI_SYMBOL_EDGEWIDTH_NORMAL = 0.4          

# --- 7. Legend Structure Configuration ---
UHI_LEGEND_TITLE = r"Mean UHI$_{\mathrm{HI90}}$ (°C)"
UHI_LEGEND_TITLE_FONTSIZE = 22
UHI_LEGEND_TEXT_FONTSIZE = 20
UHI_LEGEND_NCOL = 4             
#UHI_LEGEND_FRAMEON = True      
UHI_LEGEND_FRAMEON = False      

# Figure Margins and Gaps 
FIG_LEFT, FIG_RIGHT, FIG_BOTTOM, FIG_TOP = 0.035, 0.99, 0.11, 0.94
SUBPLOT_WSPACE, SUBPLOT_HSPACE = -0.57, 0.03

# Geographic boundary box 
MAP_LON_MIN, MAP_LON_MAX, MAP_LAT_MIN, MAP_LAT_MAX = 67.0, 98.0, 7.0, 38.0
LATLON_TICK_STEP = 10           
LON_STEP = LATLON_TICK_STEP
LAT_STEP = LATLON_TICK_STEP

# Coordinate axis labeling configurations
LATLON_LABEL_FONTSIZE = 20
LATLON_LABEL_BOLD = True
LATLON_LABEL_COLOR = "black"
SHOW_AXIS_LABELS_FIRST_COLUMN = True 
SHOW_AXIS_LABELS_LAST_ROW = True     
AXES_ANCHOR = "C"

# Map Geometry Stylings
BOUNDARY_EDGE_COLOR, BOUNDARY_FACE_COLOR, BOUNDARY_LINE_WIDTH = "black", "white", 0.8
MAP_TEXT_BOLD, MAP_TEXT_FONTSIZE = True, 20

# Geometry marker specific stylings
HI_ALPHA, HI_EDGE_COLOR, HI_EDGE_WIDTH, HI_MARKER = 0.90, "black", 0.35, "o"
UHI_EDGE_COLOR, UHI_EDGE_WIDTH, UHI_MARKER, OFFSET_LON, OFFSET_LAT = "black", 0.35, "o", 0.0, 0.0
ANNOTATE_CITIES, LABEL_FONT_SIZE = False, 14

# --- 8. Shared Colorbar Coordinates (Manual Placements) ---
COLORBAR_FONTSIZE, COLORBAR_BOLD = 20, True
MAIN_TITLE_Y, FILE_LABEL_OUTSIDE_GAP, FILE_LABEL_OUTSIDE_OFFSET = 0.995, 0.008, 0.002
GUIDE_LAYOUT = "auto"
GUIDE_CBAR_X0_HORIZONTAL, GUIDE_CBAR_Y0_HORIZONTAL, GUIDE_CBAR_W_HORIZONTAL, GUIDE_CBAR_H_HORIZONTAL = 0.137, 0.03, 0.43, 0.022
GUIDE_LEGEND_W_HORIZONTAL, GUIDE_LEGEND_H_HORIZONTAL, GUIDE_GAP_HORIZONTAL, GUIDE_SECONDARY_POSITION_HORIZONTAL, GUIDE_LEGEND_YSHIFT_HORIZONTAL = 0.25, 0.10, 0.040, "right", -0.05  

UHI_SYMBOL_BINS_AUTO_EXTEND, PRINT_UHI_SYMBOL_COVERAGE = True, True
UHI_SYMBOL_LEGEND_MARKERSIZE, UHI_SYMBOL_LEGEND_HANDLELENGTH, UHI_SYMBOL_LEGEND_COLUMNSPACING, UHI_SYMBOL_LEGEND_HANDLETEXTPAD, UHI_SYMBOL_LEGEND_LABELSPACING = 16, 0.010, 0.010, 0.010, 0.010        
UHI_LEGEND_TITLE_BOLD, UHI_LEGEND_TEXT_BOLD = True, True

# Enforcing Typography for Colorbar Ticks and Labels
COLORBAR_LABEL_FONTSIZE, COLORBAR_TICK_FONTSIZE, COLORBAR_LABEL_BOLD, COLORBAR_TICK_BOLD = 22, 20, True, True

# Output parameters
FIG_DPI, FIGSIZE_SINGLE, FIGSIZE_TWO_ROWS, SAVE_MERGED_TABLES = 750, (15, 11), (16, 11), True

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def normalize_text(x): return str(x).strip().lower()
def safe_stem(name): return re.sub(r"[^\w.\-]+", "_", str(name).strip())
def detect_column(df, candidates):
    normalized = {re.sub(r"[^a-z0-9]+", "", c.lower()): c for c in df.columns}
    for cand in candidates:
        key = re.sub(r"[^a-z0-9]+", "", cand.lower())
        if key in normalized: return normalized[key]
    return None

def format_range_label(lo, hi):
    if np.isneginf(lo): return f"< {hi:g}"
    if np.isposinf(hi): return f">= {lo:g}"
    return f"{lo:g} to {hi:g}"

def extract_file_label(source_key):
    return FILE_LABELS.get(source_key, source_key)

def order_source_keys(common_keys):
    if not SOURCE_ORDER: return list(common_keys)
    ordered, used = [], set()
    for item in SOURCE_ORDER:
        matched = None
        for key in common_keys:
            if key == item or item in key:
                matched = key; break
        if matched and matched not in used:
            ordered.append(matched)
            used.add(matched)
    for key in common_keys:
        if key not in used: ordered.append(key)
    return ordered

def draw_panel_annotations(ax, panel_index, season):
    if SHOW_PANEL_LABELS:
        add_text_box(ax, letter_for_index(panel_index), PANEL_LABEL_POSITION, PANEL_LABEL_FONTSIZE, PANEL_LABEL_BOLD)
    if SHOW_SEASON_LABEL:
        add_text_box(ax, season, SEASON_LABEL_POSITION, SEASON_LABEL_FONTSIZE, SEASON_LABEL_BOLD)

def extract_city_name(raw_name):
    if pd.isna(raw_name): return ""
    raw_name = str(raw_name).strip()
    if "__" in raw_name: raw_name = raw_name.split("__", 1)[0].strip()
    return raw_name

def canonical_metric_name(x):
    if pd.isna(x): return None
    s = str(x).strip().lower().replace(" ", "").replace(".", "_")
    metric_map = {"p90": "P90"}
    if s in metric_map: return metric_map[s]
    if re.fullmatch(r"p\d+(_\d+)?", s): return s.upper()
    return None

def pair_key_from_filename(fp):
    stem = Path(fp).stem.lower().strip().replace("(1)", "")
    stem = re.sub(r"[\s]+", "_", stem)
    canonical_patterns = [r"(historical_imdaa_1985-2014)", r"(ssp245_near_2026-2055)", r"(ssp245_far_2070-2099)", r"(ssp585_near_2026-2055)", r"(ssp585_far_2070-2099)"]
    for pat in canonical_patterns:
        m = re.search(pat, stem)
        if m: return m.group(1)
    return stem

def load_one_folder(folder, dataset_label):
    folder = Path(folder)
    if not folder.exists(): raise FileNotFoundError(f"Folder not found: {folder}")
    files = sorted(folder.glob("*.csv"))
    out = {}
    for fp in files:
        raw = pd.read_csv(fp, low_memory=False)
        source_key = pair_key_from_filename(fp)
        city_col = detect_column(raw, ["CityName", "City Name", "City", "cityname"])
        season_col = detect_column(raw, ["Season", "season"])
        metric_col = detect_column(raw, ["Metric", "metric"])
        mean_col = detect_column(raw, ["Mean", "mean"])
        median_col = detect_column(raw, ["Median", "median"])
        df = pd.DataFrame()
        df["CityName"] = raw[city_col].astype(str).str.strip().apply(extract_city_name)
        df["Season"] = raw[season_col].astype(str).str.strip().str.upper()
        df["Metric"] = raw[metric_col].astype(str).str.strip().apply(canonical_metric_name)
        df["Mean"] = pd.to_numeric(raw[mean_col], errors="coerce") if mean_col is not None else np.nan
        df["Median"] = pd.to_numeric(raw[median_col], errors="coerce") if median_col is not None else np.nan
        df = df.dropna(subset=["CityName", "Season", "Metric"]).copy()
        if df.duplicated(subset=["CityName", "Season", "Metric"], keep=False).sum() > 0:
            df = df.groupby(["CityName", "Season", "Metric"], as_index=False).agg({"Mean": "mean", "Median": "mean"})
        out[source_key] = df
    return out

def load_city_coords(csv_path):
    df = pd.read_csv(csv_path, low_memory=False)
    city_col = detect_column(df, ["CityName", "City", "cityname"])
    lat_col = detect_column(df, ["Lat", "lat", "latitude"])
    lon_col = detect_column(df, ["Lon", "lon", "longitude"])
    out = pd.DataFrame()
    out["CityName"] = df[city_col].astype(str).str.strip().apply(extract_city_name)
    out["Lat"] = pd.to_numeric(df[lat_col], errors="coerce")
    out["Lon"] = pd.to_numeric(df[lon_col], errors="coerce")
    return out.dropna().drop_duplicates(subset=["CityName"]).copy()

def read_india_boundary(boundary_file):
    gdf = gpd.read_file(boundary_file)
    if gdf.crs is None: gdf = gdf.set_crs(epsg=4326)
    else: gdf = gdf.to_crs(epsg=4326)
    gdf["__dissolve__"] = 1
    return gdf.dissolve(by="__dissolve__")

def size_scale(series, min_size=UHI_RING_MIN_SIZE, max_size=UHI_RING_MAX_SIZE, global_min=None, global_max=None):
    s = pd.to_numeric(series, errors="coerce")
    vmin, vmax = (float(global_min), float(global_max)) if global_min is not None else (s.min(), s.max())
    if np.isclose(vmin, vmax): return pd.Series(np.full(len(s), (min_size + max_size) / 2.0), index=s.index)
    return min_size + (max_size - min_size) * np.sqrt(((s - vmin) / (vmax - vmin)).clip(0, 1))

def format_lon(x, pos=None): return f"{abs(x):.1f}°E".replace(".0", "")
def format_lat(y, pos=None): return f"{abs(y):.1f}°N".replace(".0", "")

def set_map_extent(ax, boundary_gdf): 
    ax.set_xlim(MAP_LON_MIN, MAP_LON_MAX)
    ax.set_ylim(MAP_LAT_MIN, MAP_LAT_MAX)

def set_latlon_ticks(ax, boundary_gdf, lon_step=LON_STEP, lat_step=LAT_STEP, show_left=True, show_bottom=True):
    ax.set_xticks(np.arange(math.ceil(MAP_LON_MIN/lon_step)*lon_step, MAP_LON_MAX, lon_step))
    ax.set_yticks(np.arange(math.ceil(MAP_LAT_MIN/lat_step)*lat_step, MAP_LAT_MAX, lat_step))
    ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
    style_ticklabels(ax, show_left, show_bottom)

def panel_coords(p): 
    return {"upper left": (0.03, 0.97, "left", "top"), "upper right": (0.97, 0.97, "right", "top")}.get(p, (0.03, 0.97, "left", "top"))

def letter_for_index(i): 
    return string.ascii_lowercase[i % 26] + ")"

def add_text_box(ax, text, pos, fs, bold=False):
    x, y, ha, va = panel_coords(pos)
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=fs, fontweight="bold" if bold else "normal", fontname="Times New Roman", zorder=20)

def resolve_guide_layout(): 
    return GUIDE_LAYOUT if GUIDE_LAYOUT in {"vertical", "horizontal"} else ("horizontal" if GRID_ORIENTATION == "seasons_rows_files_cols" else "vertical")

def place_outside_label(fig, ax, text, position="top", fontsize=12, bold=True, offset=0.012):
    bbox = ax.get_position()
    x, y, ha, va, rot = ((bbox.x0+bbox.x1)/2.0, bbox.y1+offset, "center", "bottom", 0) if position=="top" else (bbox.x0-offset, (bbox.y0+bbox.y1)/2.0, "right", "center", 90)
    fig.text(x, y, text, ha=ha, va=va, rotation=rot, fontsize=fontsize, fontweight="bold" if bold else "normal", fontname="Times New Roman", transform=fig.transFigure, zorder=30)

def style_ticklabels(ax, sl=True, sb=True):
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontsize(LATLON_LABEL_FONTSIZE)
        label.set_fontweight("bold" if LATLON_LABEL_BOLD else "normal")
        label.set_color(LATLON_LABEL_COLOR)
        label.set_fontname("Times New Roman") 
    ax.tick_params(axis="x", labelbottom=sb, bottom=True)
    ax.tick_params(axis="y", labelleft=sl, left=True)

def add_size_legend(global_min, global_max, mode="rings"):
    if mode == "rings":
        ref_vals = np.linspace(global_min, global_max, UHI_RING_LEGEND_LEVELS)
        handles, labels = [], []
        for v in ref_vals:
            s = size_scale(pd.Series([v]), global_min=global_min, global_max=global_max).iloc[0]
            handles.append(Line2D([0], [0], marker="o", linestyle="None", markerfacecolor="none", markeredgecolor=UHI_EDGE_COLOR, markersize=max(2.0, math.sqrt(s)*UHI_RING_LEGEND_MARKERSIZE_SCALE)))
            labels.append(f"{v:.2f}")
    else:
        handles, labels = [], []
        bins = np.asarray(UHI_SYMBOL_BINS, dtype=float).copy()
        if UHI_SYMBOL_BINS_AUTO_EXTEND:
            bins[0], bins[-1] = -np.inf, np.inf
        for i in range(len(bins) - 1):
            f_color = UHI_SYMBOL_FACE_COLORS[i] if i < len(UHI_SYMBOL_FACE_COLORS) else "none"
            handles.append(Line2D([0], [0], marker=UHI_SYMBOL_MARKERS[i], linestyle="None", markerfacecolor=f_color, markeredgecolor=UHI_EDGE_COLOR, markersize=UHI_SYMBOL_LEGEND_MARKERSIZE, markeredgewidth=UHI_SYMBOL_EDGEWIDTH_NORMAL))
            labels.append(format_range_label(bins[i], bins[i+1]))
    return handles, labels

def get_metric_scales(uhi_df, hi_df, metric, value_column):
    hi_vals = pd.to_numeric(hi_df[value_column] if GLOBAL_SCALE_MODE=="all_metrics" else hi_df.loc[hi_df["Metric"]==metric, value_column], errors="coerce").dropna()
    uhi_vals = pd.to_numeric(uhi_df[value_column] if GLOBAL_SCALE_MODE=="all_metrics" else uhi_df.loc[uhi_df["Metric"]==metric, value_column], errors="coerce").dropna()
    return {"hi_norm": Normalize(vmin=hi_vals.min(), vmax=hi_vals.max()), "uhi_min": uhi_vals.min(), "uhi_max": uhi_vals.max()}

def draw_map_base(ax, boundary_gdf, sl=True, sb=True):
    boundary_gdf.plot(ax=ax, facecolor=BOUNDARY_FACE_COLOR, edgecolor=BOUNDARY_EDGE_COLOR, linewidth=BOUNDARY_LINE_WIDTH)
    set_map_extent(ax, boundary_gdf)
    set_latlon_ticks(ax, boundary_gdf, show_left=sl, show_bottom=sb)
    ax.set_aspect("equal", adjustable="box")
    ax.set_anchor(AXES_ANCHOR)

def draw_hi_layer(ax, hi_df, hi_norm):
    ax.scatter(hi_df["Lon"], hi_df["Lat"], s=HI_FIXED_POINT_SIZE, c=hi_df["PlotValue"], cmap=HI_CMAP, norm=hi_norm, marker=HI_MARKER, alpha=HI_ALPHA, edgecolors=HI_EDGE_COLOR, linewidths=HI_EDGE_WIDTH, zorder=4)

def draw_uhi_layer(ax, uhi_df, global_min, global_max):
    if UHI_REPRESENTATION_MODE == "rings":
        ax.scatter(uhi_df["Lon"]-OFFSET_LON, uhi_df["Lat"]+OFFSET_LAT, s=size_scale(uhi_df["PlotValue"], global_min=global_min, global_max=global_max), facecolors=UHI_FACE_COLOR, edgecolors=UHI_EDGE_COLOR, marker=UHI_MARKER, linewidths=UHI_EDGE_WIDTH, zorder=5)
        return
    bins = np.asarray(UHI_SYMBOL_BINS, dtype=float).copy()
    if UHI_SYMBOL_BINS_AUTO_EXTEND: 
        bins[0], bins[-1] = -np.inf, np.inf
    cats = pd.cut(uhi_df["PlotValue"], bins=bins, include_lowest=True)
    for i, interval in enumerate(cats.cat.categories):
        mask = cats == interval
        if not mask.any(): continue
        f_color = UHI_SYMBOL_FACE_COLORS[i] if i < len(UHI_SYMBOL_FACE_COLORS) else "none"
        ax.scatter(uhi_df.loc[mask, "Lon"]-OFFSET_LON, uhi_df.loc[mask, "Lat"]+OFFSET_LAT, s=UHI_SYMBOL_POINT_SIZE, marker=UHI_SYMBOL_MARKERS[i], facecolors=f_color, edgecolors=UHI_EDGE_COLOR, linewidths=UHI_SYMBOL_EDGEWIDTH_NORMAL, zorder=5)

def prepare_plot_data(uhi_df, hi_df, coords_df, season, metric, value_column):
    u = uhi_df[(uhi_df["Season"] == season) & (uhi_df["Metric"] == metric)].merge(coords_df, on="CityName", how="left").dropna(subset=["Lat", "Lon"])
    h = hi_df[(hi_df["Season"] == season) & (hi_df["Metric"] == metric)].merge(coords_df, on="CityName", how="left").dropna(subset=["Lat", "Lon"])
    u["PlotValue"], h["PlotValue"] = pd.to_numeric(u[value_column], errors="coerce"), pd.to_numeric(h[value_column], errors="coerce")
    u, h = u.dropna(subset=["PlotValue"]), h.dropna(subset=["PlotValue"])
    cc = sorted(set(u["CityName"]).intersection(set(h["CityName"])))
    return u[u["CityName"].isin(cc)], h[h["CityName"].isin(cc)], pd.merge(u, h, on=["CityName", "Lat", "Lon", "Season", "Metric"])

def add_shared_guides(fig, hi_norm, global_min, global_max):
    layout = resolve_guide_layout()
    handles, labels = add_size_legend(global_min, global_max, mode=UHI_REPRESENTATION_MODE)
    sm = ScalarMappable(norm=hi_norm, cmap=HI_CMAP)
    sm.set_array([])
    
    cax = fig.add_axes([GUIDE_CBAR_X0_HORIZONTAL, GUIDE_CBAR_Y0_HORIZONTAL, GUIDE_CBAR_W_HORIZONTAL, GUIDE_CBAR_H_HORIZONTAL])
    lax = fig.add_axes([GUIDE_CBAR_X0_HORIZONTAL + GUIDE_CBAR_W_HORIZONTAL + GUIDE_GAP_HORIZONTAL, GUIDE_CBAR_Y0_HORIZONTAL + GUIDE_LEGEND_YSHIFT_HORIZONTAL, GUIDE_LEGEND_W_HORIZONTAL, GUIDE_LEGEND_H_HORIZONTAL])
    
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    
    cbar.set_label("Mean HI$_{90}$ (°C)", fontsize=COLORBAR_LABEL_FONTSIZE, fontweight="bold" if COLORBAR_LABEL_BOLD else "normal", fontname="Times New Roman")
    cbar.ax.tick_params(labelsize=COLORBAR_TICK_FONTSIZE)
    for t in cbar.ax.get_xticklabels() + cbar.ax.get_yticklabels():
        t.set_fontname("Times New Roman")
        t.set_fontweight("bold" if COLORBAR_TICK_BOLD else "normal")
        
    lax.axis("off")
    
    legend_prop = {'family': 'Times New Roman', 'size': UHI_LEGEND_TEXT_FONTSIZE, 'weight': 'bold' if UHI_LEGEND_TEXT_BOLD else 'normal'}
    leg = lax.legend(handles, labels, title=UHI_LEGEND_TITLE, loc="center", frameon=UHI_LEGEND_FRAMEON, ncol=UHI_LEGEND_NCOL, prop=legend_prop)
    
    leg.get_title().set_fontname("Times New Roman")
    leg.get_title().set_fontsize(UHI_LEGEND_TITLE_FONTSIZE) 
    leg.get_title().set_fontweight("bold" if UHI_LEGEND_TITLE_BOLD else "normal")

def build_grid_figure(source_plan, uhi_files, hi_files, coords_df, boundary_gdf, metric, value_column):
    uhi_all = pd.concat(list(uhi_files.values()), ignore_index=True)
    hi_all = pd.concat(list(hi_files.values()), ignore_index=True)
    sc = get_metric_scales(uhi_all, hi_all, metric, value_column)
    
    row_keys, col_keys = (SEASONS_TO_PLOT, source_plan) if GRID_ORIENTATION == "seasons_rows_files_cols" else (source_plan, SEASONS_TO_PLOT)
    nrows, ncols = len(row_keys), len(col_keys)
    fig, axes = plt.subplots(nrows, ncols, figsize=(max(4.8 * ncols, 12), max(4.3 * nrows, 8)))
    axes = np.array([[axes]]) if nrows==1 and ncols==1 else (np.array([axes]) if nrows==1 else (np.array([[ax] for ax in axes]) if ncols==1 else axes))
    fig.subplots_adjust(left=FIG_LEFT, right=FIG_RIGHT, bottom=FIG_BOTTOM, top=FIG_TOP, wspace=SUBPLOT_WSPACE, hspace=SUBPLOT_HSPACE)
    
    panel_index = 0
    for i, rk in enumerate(row_keys):
        for j, ck in enumerate(col_keys):
            season, sk = (rk, ck) if GRID_ORIENTATION == "seasons_rows_files_cols" else (ck, rk)
            u_df = uhi_files[sk]; h_df = hi_files[sk]
            u_plot, h_plot, merged = prepare_plot_data(u_df, h_df, coords_df, season, metric, value_column)
            ax = axes[i, j]
            if merged.empty: 
                ax.set_axis_off()
                continue
            draw_map_base(ax, boundary_gdf, j==0, i==nrows-1)
            draw_hi_layer(ax, h_plot, sc["hi_norm"])
            draw_uhi_layer(ax, u_plot, sc["uhi_min"], sc["uhi_max"])
            draw_panel_annotations(ax, panel_index, season)
            panel_index += 1
            
    if SHOW_FILE_LABELS:
        for j, sk in enumerate(col_keys if GRID_ORIENTATION == "seasons_rows_files_cols" else row_keys):
            place_outside_label(fig, axes[0, j], extract_file_label(sk), "top", FILE_LABEL_FONTSIZE, FILE_LABEL_BOLD)
            
    add_shared_guides(fig, sc["hi_norm"], sc["uhi_min"], sc["uhi_max"])
        
    base_filename = f"{safe_stem(MAIN_TITLE if MAIN_TITLE else 'Map')}_{metric}_{value_column}"
    for ext in ['png', 'svg', 'pdf', 'jpg', 'jpeg']:
        out_path = OUTPUT_DIR / f"{base_filename}.{ext}"
        fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

def main():
    uhi_files = load_one_folder(UHI_INPUT_DIR, "UHI")
    hi_files = load_one_folder(HI_INPUT_DIR, "HI")
    coords_df = load_city_coords(CITY_COORDS_CSV)
    boundary_gdf = read_india_boundary(INDIA_BOUNDARY_FILE)
    sp = order_source_keys(sorted(set(uhi_files.keys()).intersection(hi_files.keys())))
    for metric in METRICS_TO_PLOT:
        build_grid_figure(sp, uhi_files, hi_files, coords_df, boundary_gdf, metric, VALUE_COLUMN)
    print("Done processing map outputs.")

if __name__ == "__main__":
    with warnings.catch_warnings(): 
        warnings.simplefilter("ignore")
        main()

Done processing map outputs.
